# 04 — LSTM Deep Learning Model

Driver behaviour classification from raw time-series windows.
Architecture options: **LSTM** or **CNN-LSTM hybrid**.

> Requires PyTorch: `pip install torch torchvision`

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
from pathlib import Path

from data_loader import UAHDriveSetLoader
from features import build_sequence_dataset
from models import DriverBehaviorLSTM, DriverBehaviorCNN_LSTM
from evaluate import compute_metrics, save_metrics, plot_confusion_matrix, plot_training_history

DATA_DIR     = Path('../data/raw/UAH-DRIVESET-v1')
FIG_DIR      = Path('../results/figures'); FIG_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = Path('../results/metrics/metrics.json')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

WINDOW_S  = 5.0
OVERLAP_S = 2.0
BATCH     = 64
EPOCHS    = 40
LR        = 1e-3

In [ ]:
loader = UAHDriveSetLoader(DATA_DIR, merge=True).load(verbose=False)

SEQ_COLS = ['acc_x_kf', 'acc_y_kf', 'acc_z_kf', 'roll', 'pitch', 'yaw']

X, y = build_sequence_dataset(loader.trips, SEQ_COLS, window_s=WINDOW_S, overlap_s=OVERLAP_S)
print(f'X shape: {X.shape}  y shape: {y.shape}')
print(f'Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}')

In [ ]:
# Normalisation (per-channel)
mean = X.mean(axis=(0, 1), keepdims=True)
std  = X.std(axis=(0, 1), keepdims=True) + 1e-8
X_norm = (X - mean) / std

X_t = torch.FloatTensor(X_norm)
y_t = torch.LongTensor(y)

dataset  = TensorDataset(X_t, y_t)
n_train  = int(0.8 * len(dataset))
n_val    = len(dataset) - n_train
train_ds, val_ds = random_split(dataset, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)
print(f'Train: {n_train}  Validation: {n_val}')

## Model Definition & Training

In [ ]:
model = DriverBehaviorLSTM(
    input_size=X.shape[2],
    hidden_size=128,
    num_layers=2,
    n_classes=3,
    dropout=0.3,
).to(DEVICE)

# Class weights (for class imbalance)
class_counts = np.bincount(y)
weights = torch.FloatTensor(1.0 / (class_counts + 1e-6)).to(DEVICE)
weights = weights / weights.sum() * len(class_counts)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')

In [ ]:
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
best_val_acc = 0.0

for epoch in range(EPOCHS):
    # --- Training ---
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss    += loss.item() * len(yb)
        t_correct += (logits.argmax(1) == yb).sum().item()
        t_total   += len(yb)

    scheduler.step()

    # --- Validation ---
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits  = model(xb)
            v_loss += criterion(logits, yb).item() * len(yb)
            v_correct += (logits.argmax(1) == yb).sum().item()
            v_total   += len(yb)

    ta = t_correct / t_total; va = v_correct / v_total
    tl = t_loss / t_total;    vl = v_loss / v_total
    train_losses.append(tl); val_losses.append(vl)
    train_accs.append(ta);   val_accs.append(va)

    if va > best_val_acc:
        best_val_acc = va
        torch.save(model.state_dict(), '../results/best_lstm.pth')

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | TrainLoss={tl:.4f} ValLoss={vl:.4f} | TrainAcc={ta:.3f} ValAcc={va:.3f}')

print(f'\nBest validation accuracy: {best_val_acc:.4f}')

In [ ]:
plot_training_history(
    train_losses, val_losses, train_accs, val_accs,
    title='LSTM Training History',
    save_path=FIG_DIR / 'lstm_training_history.png'
)
plt.show()

## Evaluation

In [ ]:
model.load_state_dict(torch.load('../results/best_lstm.pth', map_location=DEVICE))
model.eval()

all_preds, all_labels, all_proba = [], [], []
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        proba  = torch.softmax(logits, dim=1).cpu().numpy()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(yb.numpy())
        all_proba.extend(proba)

y_true  = np.array(all_labels)
y_pred  = np.array(all_preds)
y_proba = np.array(all_proba)

m = compute_metrics(y_true, y_pred, y_proba)
print(m['report'])
save_metrics(m, METRICS_PATH, model_name='LSTM')

plot_confusion_matrix(y_true, y_pred, title='LSTM — Confusion Matrix',
                      save_path=FIG_DIR / 'cm_lstm.png')
plt.show()